# Kothiq — free ASR + TTS server

Runs open-source Bangla ASR (Whisper via `faster-whisper`) and TTS (Meta MMS-TTS Bengali)
on Colab's free GPU, and exposes them as a tiny HTTP API through a free `cloudflared`
tunnel — no ngrok account needed.

**Runtime > Change runtime type > GPU (T4)** before running these cells.

Run all cells top to bottom. The last cell prints a public URL — paste it into your
local `backend/.env` as `COLAB_ASR_URL` and `COLAB_TTS_URL` (same URL for both), then
set `ASR_PROVIDER=colab` and `TTS_PROVIDER=colab` there too.

This URL changes every time you rerun the notebook (free tunnel, no fixed address) —
update `.env` again each session and restart your local backend.

In [ ]:
!pip install -q faster-whisper transformers accelerate soundfile fastapi "uvicorn[standard]" python-multipart

In [ ]:
# Load ASR (faster-whisper). "medium" is a good speed/quality balance on a free T4.
# Bump to "large-v3" if you have more GPU time/patience and want better accuracy.
from faster_whisper import WhisperModel

WHISPER_SIZE = "medium"
whisper_model = WhisperModel(WHISPER_SIZE, device="cuda", compute_type="float16")
print(f"Loaded whisper '{WHISPER_SIZE}' on GPU")

In [ ]:
# Load TTS (Meta MMS-TTS, Bengali). Free, open, real Bengali voice — accent leans
# generic/Indian-Bengali rather than Dhaka, and sounds more robotic than commercial
# TTS. That's expected for a free Phase-1 bridge; swap for your own trained voice later.
import torch
from transformers import VitsModel, AutoTokenizer

tts_model = VitsModel.from_pretrained("facebook/mms-tts-ben").to("cuda")
tts_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-ben")
print("Loaded facebook/mms-tts-ben on GPU")

In [ ]:
import io
import tempfile

import soundfile as sf
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import Response

app = FastAPI(title="Kothiq free ASR+TTS server")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/transcribe")
async def transcribe(audio: UploadFile = File(...)):
    data = await audio.read()
    with tempfile.NamedTemporaryFile(suffix=".webm") as f:
        f.write(data)
        f.flush()
        segments, _info = whisper_model.transcribe(f.name, language="bn")
        text = " ".join(seg.text for seg in segments).strip()
    return {"text": text}


@app.post("/synthesize")
async def synthesize(text: str = Form(...)):
    inputs = tts_tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        waveform = tts_model(**inputs).waveform
    wav = waveform.squeeze().cpu().numpy()

    buf = io.BytesIO()
    sf.write(buf, wav, tts_model.config.sampling_rate, format="WAV")
    buf.seek(0)
    return Response(content=buf.read(), media_type="audio/wav")

In [ ]:
# Start the API server in a background thread so this notebook stays interactive.
import threading

import uvicorn


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=_run_server, daemon=True).start()
print("Server starting on port 8000...")

In [ ]:
# Expose port 8000 publicly via a free cloudflared "quick tunnel" (no account needed).
import re
import subprocess
import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

time.sleep(2)  # give the local server a moment to bind the port

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n=== Your API is live at ===")
print(public_url)
print("\nPaste into your local backend/.env:")
print(f"COLAB_ASR_URL={public_url}")
print(f"COLAB_TTS_URL={public_url}")
print("ASR_PROVIDER=colab")
print("TTS_PROVIDER=colab")